In [1]:
# Importing modules
import json
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, auc
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Reading lipid df
df = pd.read_csv("../data/final_data.tsv")

# Discovery data
train_df = df.loc[df["Datatype"]=="Training"]
X_train = train_df.drop(["Class", "Samples", "Datatype"], axis=1)
y_train = train_df["Class"]

In [3]:
# Selected features
f1 = list(pd.read_excel("../data/1_NAFLD_Analysis_Top_1_Features_List.xlsx", sheet_name="1_NF_NH_Topmost_2_List")["Lipid"])
f2 = list(pd.read_excel("../data/1_NAFLD_Analysis_Top_1_Features_List.xlsx", sheet_name="2_NS_NH_Topmost_2_List")["Lipid"])
f3 = list(pd.read_excel("../data/1_NAFLD_Analysis_Top_1_Features_List.xlsx", sheet_name="3_NS_NF_Topmost_2_List")["Lipid"])
selected_features = list(set(f1+f2+f3))

In [4]:
# Subsetting feature matrix with selected features
X_train = pd.DataFrame(X_train, columns=selected_features)

In [5]:
# Label encoding
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)

In [6]:
# Scaling the data
sds = StandardScaler()
X_train_orig, y_train_orig = X_train, y_train
X_train = pd.DataFrame(sds.fit_transform(X_train), columns=X_train.columns)
X_train_orig = pd.DataFrame(sds.fit_transform(X_train_orig), columns=X_train_orig.columns)

In [7]:
# Models
models = {
    'RandomForestClassifier': RandomForestClassifier(random_state=42),
}

# Parameters
params = {
    'RandomForestClassifier': {
        'n_estimators': [20, 50, 100],
        'max_depth': [2, 3, 5, None],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 'log2'],
        'bootstrap': [True]
    }
}


best_models = {}

# Model selection and hyperparameter tuning
for name, model in models.items():
    
    # Training using Grid Search with 5 fold CV
    model = GridSearchCV(model, params[name], cv=5, scoring='roc_auc', refit=True)
    model.fit(X_train.to_numpy(), y_train)
    
    # Model
    model = model.best_estimator_
    best_models[name] = model

In [8]:
names = []
accuracy = []
precision = []
recall = []
f1score = []

for m in models.keys():
    model = best_models[m]
    y_pred = model.predict(X_train_orig.to_numpy())
    names.append(m)
    accuracy.append(round(accuracy_score(y_train_orig, y_pred), 2))
    precision.append(round(precision_score(y_train_orig, y_pred, average='weighted'), 2))
    recall.append(round(recall_score(y_train_orig, y_pred, average='weighted'),2))
    f1score.append(round(f1_score(y_train_orig, y_pred, average='weighted'),2))
    
# Table showing models performance on Validation Set
print("Performance on Discovery Set:")
pd.DataFrame({"Model":names, "Accuracy":accuracy, "Precision":precision, "Recall":recall, "F1-Score":f1score})

Performance on Discovery Set:


,Model,Accuracy,Precision,Recall,F1-Score
0,RandomForestClassifier,0.96,0.96,0.96,0.96


In [9]:
best_models["RandomForestClassifier"].get_params()

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'class_weight': None,
 'criterion': 'gini',
 'max_depth': 2,
 'max_features': 'sqrt',
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'n_estimators': 20,
 'n_jobs': None,
 'oob_score': False,
 'random_state': 42,
 'verbose': 0,
 'warm_start': False}

In [10]:
# Saving best params
with open("../model/best_params.json", "w") as f:
    best_params = best_models["RandomForestClassifier"].get_params()
    json.dump(best_params, f)